# Expert draft grades

The expert side of the comparison needs the same thing the fan side has: an
instant, per-pick reaction recorded before any NFL outcome existed. That rules
out most of the draft-grade industry, which works at team level or regrades in
hindsight:

- **PFF, NFL.com, CBS, SI** publish per-*team* grades, or per-pick grades for
  round 1 only.
- **Pro Football Network** grades every pick but is spread across 30+ pages a
  year behind a scrape-blocker.
- **WalterFootball** posts a letter grade for **every pick, all seven rounds,
  minutes after each selection** — and has done so with one grader (Walt) and
  one scale for all six drafts in this study. Pages: `nfldraftgradesYYYY*.php`
  (current year unsuffixed), archived in `data/raw/walterfootball/`.

WalterFootball is therefore the expert measure. The trade-off is explicit:
this is one analyst's instant grade, not a multi-outlet consensus. What it
buys is methodological symmetry — the same grader, scale, and timing across
every pick and every class, which no consensus source offers for rounds 2–7.

**The re-grade trap.** Archived pages carry a later "One-Year Grade" above the
original write-up. Quentin Johnston's page shows why using it would be fatal:
original A−, one-year F. Only the original draft-night grade is parsed here.

### Parsing

Entries follow a `Team: Player, POS, School -` header with the grade on the
next line. Anchoring on the *header* (rather than the grade line) sidesteps
the two formatting hazards found in the raw pages:

- The one-year re-grade lines never match a header, so the grade taken is
  always the original.
- Kicker and punter picks get joke grades — `O'MILLEN GRADE`, `MILLEN-19
  GRADE`, `WHAT BIG KIELBASAS MILLEN GRADE`, `O'BRIEN Grade` — Walt's
  below-the-scale insult grades, named for Matt Millen and Bill O'Brien.
  All are classified as `MILLEN`, plus stray typos (`B Grades`, double
  spaces) that a strict pattern would drop.

In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

WF_DIR = Path("..") / "data" / "raw" / "walterfootball"

HEADER = re.compile(r".+:.+,.+-$")
LETTER = re.compile(r"\b([A-F][+-]?)\s+Grades?$", re.I)

def page_lines(path):
    html = path.read_text(errors="replace")
    body = re.sub(r"<(script|style).*?</\1>", " ", html, flags=re.S)
    body = re.sub(r"<[^>]+>", "\n", body)
    return [l.strip() for l in body.split("\n") if l.strip()]

def classify_grade(line):
    if len(line) > 80:
        return None
    up = line.upper()
    if "MILLEN" in up or "O'BRIEN" in up:
        return "MILLEN"
    m = LETTER.search(line)
    return m.group(1).upper() if m else None

rows, unparsed = [], []
for path in sorted(WF_DIR.glob("*.html")):
    season = int(path.stem.split("_")[0])
    rnd = 1 if path.stem.endswith(("r1a", "r1b")) else int(path.stem[-1])
    lines = page_lines(path)
    for i, line in enumerate(lines):
        if not HEADER.match(line.rstrip()) and not (
                line == "-" and i and HEADER.match((lines[i-1] + " -"))):
            continue
        header = lines[i-1] if line == "-" else line
        if line == "-" and not header.rstrip().endswith(","):
            header = lines[i-1]
        grade_line = lines[i+1] if i + 1 < len(lines) else ""
        grade = classify_grade(grade_line)
        if grade is None:
            continue
        player = header.split(":", 1)[1].split(",")[0].strip()
        rows.append({"season": season, "round": rnd, "order": len(rows),
                     "wf_player": player, "wf_grade": grade})

# headers whose dash sits on its own line ('Team: Player, CB, Florida' + '-')
for path in sorted(WF_DIR.glob("*.html")):
    season = int(path.stem.split("_")[0])
    rnd = 1 if path.stem.endswith(("r1a", "r1b")) else int(path.stem[-1])
    lines = page_lines(path)
    for i, line in enumerate(lines):
        if line != "-" or i == 0 or i + 1 >= len(lines):
            continue
        header = lines[i-1]
        if ":" not in header or "," not in header:
            continue
        grade = classify_grade(lines[i+1])
        if grade is None:
            continue
        player = header.split(":", 1)[1].split(",")[0].strip()
        rows.append({"season": season, "round": rnd, "order": len(rows),
                     "wf_player": player, "wf_grade": grade})

wf = pd.DataFrame(rows).drop_duplicates(["season", "round", "wf_player", "wf_grade"])
print("parsed grades:", len(wf))
print(wf.groupby("season").size().to_string())
print("MILLEN/O'BRIEN grades:", (wf.wf_grade == "MILLEN").sum())

parsed grades: 1551
season
2021    259
2022    262
2023    259
2024    257
2025    257
2026    257
MILLEN/O'BRIEN grades: 33


### Joining grades to picks

The join reuses the name normalization from `04_match_threads.ipynb` and
degrades gracefully through tiers, each requiring uniqueness on both sides
within its key:

1. exact normalized full name (+ round, then season-wide)
2. compacted name, no spaces — catches `J.C. Latham` vs `JC Latham` and
   hyphen/space variation
3. surname (+ round, then season-wide) — catches nicknames (`Sauce`/Ahmad
   Gardner) and first-name variants
4. fuzzy compacted name — catches misspellings (`Christiansen`,
   `Tippman`, `LaJohntay Webster`)
5. an explicit fix-up table for the three cases nothing generic should
   guess: a post-draft name change (Joe Tryon → Tryon-Shoyinka), a
   mid-name comma (`Bailey, Zappe`), and `Nicholas Garguilo` for Nick
   Gargiulo

The one true ambiguity — two Byron Youngs drafted in round 3 of 2023 — is
resolved by page order: grades appear in pick order within a round.

In [2]:
import difflib

SUFFIXES = {"jr", "sr", "ii", "iii", "iv", "v"}

def normalize(text):
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def keys(full_name):
    tokens = [t for t in normalize(full_name).replace("-", " ").split()
              if t not in SUFFIXES]
    full = " ".join(tokens)
    return full, full.replace(" ", ""), (tokens[-1] if tokens else "")

FIXES = {
    (2021, "Joe Tryon"): "Joe Tryon-Shoyinka",
    (2022, "Bailey"): "Bailey Zappe",
    (2024, "Nicholas Garguilo"): "Nick Gargiulo",
}
wf["wf_player"] = wf.apply(
    lambda r: FIXES.get((r.season, r.wf_player), r.wf_player), axis=1)

outcomes = pd.read_csv("../data/processed/draft_outcomes_2021_2025.csv")
picks_2026 = pd.read_csv("../data/processed/draft_2026_picks.csv")
picks = pd.concat([outcomes, picks_2026], ignore_index=True)

picks[["name_norm", "compact", "surname"]] = picks["pfr_player_name"].apply(
    lambda n: pd.Series(keys(n)))
wf[["name_norm", "compact", "surname"]] = wf["wf_player"].apply(
    lambda n: pd.Series(keys(n)))

picks["wf_grade"] = None
picks["match_method"] = None
wf["used"] = False

def assign(cols, label):
    open_p = picks[picks.wf_grade.isna()]
    open_w = wf[~wf.used]
    pk = open_p.groupby(cols).filter(lambda g: len(g) == 1)
    wk = open_w.groupby(cols).filter(lambda g: len(g) == 1)
    merged = pk.reset_index().merge(
        wk.reset_index(), on=cols, suffixes=("", "_w"))
    for r in merged.itertuples():
        picks.at[r.index, "wf_grade"] = r.wf_grade_w
        picks.at[r.index, "match_method"] = label
        wf.at[r.index_w, "used"] = True
    print(f"{label:18} +{len(merged):4}  total {picks.wf_grade.notna().sum()}/{len(picks)}")

assign(["season", "round", "name_norm"], "name+round")
assign(["season", "name_norm"], "name")
assign(["season", "round", "compact"], "compact+round")
assign(["season", "compact"], "compact")
assign(["season", "round", "surname"], "surname+round")
assign(["season", "surname"], "surname")

# same-name duplicates, zipped in pick order (the two 2023 Byron Youngs)
for (s, rnd, nm), grp in wf[~wf.used].groupby(["season", "round", "name_norm"]):
    cand = picks[picks.wf_grade.isna() & (picks.season == s)
                 & (picks["round"] == rnd) & (picks.name_norm == nm)]
    if len(grp) == len(cand) > 0:
        for wi, pi in zip(grp.sort_values("order").index,
                          cand.sort_values("pick").index):
            picks.at[pi, "wf_grade"] = wf.at[wi, "wf_grade"]
            picks.at[pi, "match_method"] = "order-zip"
            wf.at[wi, "used"] = True
            print(f"order-zip: {s} R{rnd} {nm} -> {wf.at[wi, 'wf_grade']}")

# fuzzy tier for misspellings
for pi, prow in picks[picks.wf_grade.isna()].iterrows():
    pool = wf[~wf.used & (wf.season == prow.season)]
    if pool.empty:
        continue
    best = difflib.get_close_matches(prow.compact, pool.compact.tolist(),
                                     n=1, cutoff=0.75)
    if best:
        hits = pool[pool.compact == best[0]]
        if len(hits) == 1:
            wi = hits.index[0]
            picks.at[pi, "wf_grade"] = wf.at[wi, "wf_grade"]
            picks.at[pi, "match_method"] = "fuzzy"
            wf.at[wi, "used"] = True
            print(f"fuzzy: {prow.season} {prow.pfr_player_name!r} <- "
                  f"{wf.at[wi, 'wf_player']!r} ({wf.at[wi, 'wf_grade']})")

missing = picks[picks.wf_grade.isna()]
print(f"\npicks without a grade: {len(missing)}")
if len(missing):
    print(missing[["season", "team", "round", "pick", "pfr_player_name"]]
          .to_string(index=False))
print("unused WF rows:", (~wf.used).sum())

name+round         +1463  total 1463/1551
name               +   0  total 1463/1551
compact+round      +  27  total 1490/1551
compact            +   0  total 1490/1551
surname+round      +  43  total 1533/1551
surname            +   0  total 1533/1551
order-zip: 2023 R3 byron young -> B
order-zip: 2023 R3 byron young -> A-
fuzzy: 2021 'Brady Christensen' <- 'Brady Christiansen' (B+)
fuzzy: 2021 'Jimmy Morrissey' <- 'Jimmy Morissey' (A)
fuzzy: 2022 'Kyle Philips' <- 'Kyle Phillips' (B)
fuzzy: 2022 'Andrew Stueber' <- 'Andrew Steuber' (B+)
fuzzy: 2023 'Quentin Johnston' <- 'Quentin Johnson' (A-)
fuzzy: 2023 'Joe Tippmann' <- 'Joe Tippman' (A)
fuzzy: 2023 'Chamarri Conner' <- 'Chamarri Connor' (C-)
fuzzy: 2023 'Jovaughn Gwyn' <- 'Jovaughn Gwynn' (C+)
fuzzy: 2024 'Mike Sainristil' <- 'Mike Sainristill' (B)
fuzzy: 2024 'Sedrick Van Pran-Granger' <- 'Sedrick Van Pran' (A)
fuzzy: 2025 'Oronde Gadsden II' <- 'Orande Gadsen II' (B+)
fuzzy: 2025 'LaJohntay Wester' <- 'LaJohntay Webster' (C)
fuzz

### From letters to numbers

Letter grades map to the familiar 4.0 scale. `MILLEN` (and its `O'BRIEN`
cousin) sits one full F–D− step *below* F, preserving the fact that Walt
reserves it for picks he considers categorically worse than a normal failure —
almost always kickers and punters taken on day two or three.

The absolute values barely matter: like DrAV and fan sentiment, grades are
**z-scored within each draft class**, so the model only ever sees how far a
grade sits from that year's average. What the mapping must preserve is order.

In [3]:
GRADE_POINTS = {
    "A+": 4.3, "A": 4.0, "A-": 3.7,
    "B+": 3.3, "B": 3.0, "B-": 2.7,
    "C+": 2.3, "C": 2.0, "C-": 1.7,
    "D+": 1.3, "D": 1.0, "D-": 0.7,
    "F": 0.0, "MILLEN": -0.7,
}
picks["grade_points"] = picks["wf_grade"].map(GRADE_POINTS)
picks["grade_z"] = (picks.groupby("season")["grade_points"]
                    .transform(lambda s: (s - s.mean()) / s.std()))

print("grade distribution:")
print(picks["wf_grade"].value_counts().reindex(GRADE_POINTS).to_string())
print("\nmean grade points by season (drift the z-score removes):")
print(picks.groupby("season")["grade_points"].mean().round(2).to_string())
print("\nmean grade points by round:")
print(picks.groupby("round")["grade_points"].mean().round(2).to_string())

grade distribution:
wf_grade
A+        147.0
A         198.0
A-        139.0
B+        252.0
B         327.0
B-         95.0
C+         65.0
C         140.0
C-         68.0
D+          NaN
D          70.0
D-          3.0
F          14.0
MILLEN     33.0

mean grade points by season (drift the z-score removes):
season
2021    2.82
2022    2.95
2023    2.99
2024    3.05
2025    3.10
2026    2.89

mean grade points by round:
round
1    3.02
2    3.02
3    2.74
4    2.83
5    3.03
6    3.01
7    3.13


In [4]:
out = picks[["season", "team", "round", "pick", "pfr_player_name",
             "wf_grade", "grade_points", "grade_z", "match_method"]]
out_path = Path("..") / "data" / "processed" / "expert_grades.csv"
out.to_csv(out_path, index=False)
print(f"wrote {len(out)} rows ({out.wf_grade.notna().sum()} graded) to {out_path}")

wrote 1551 rows (1551 graded) to ../data/processed/expert_grades.csv
